
# Group-Aware Stratified Data Split — AeroSound

## 1. Dataset and objective

The objective of this notebook is to create reproducible training, validation, and test splits for the AeroSound acoustic anomaly detection project.

The MIMII dataset contains 54,057 audio recordings covering:

- Four machine types: fan, pump, slider, and valve.
- Three signal-to-noise ratio (SNR) conditions: -6 dB, 0 dB, and +6 dB.
- Two labels: `normal` and `abnormal`.

The splitting strategy aims to:

1. Prevent potentially related recordings from appearing in different datasets.
2. Preserve the distribution of normal and abnormal recordings.
3. Maintain a representative distribution across machine types.
4. Ensure that all models can be evaluated using the same data partitions.

This notebook operates on metadata only. Audio files are not loaded or processed.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


df = pd.read_csv('../raw_data/metadata.csv')

FileNotFoundError: [Errno 2] No such file or directory: '../raw_data/metadata.csv'

In [2]:
print(df.shape)

pd.crosstab(df['snr'], df['machine'])

(54057, 7)


machine,fan,pump,slider,valve
snr,,,,
-6,5550,4205,4094,4170
0,5550,4205,4094,4170
6,5550,4205,4094,4170


In [3]:
print(pd.crosstab(df['label'], df['machine']))
pd.crosstab([df['machine'],df['id']], df['label'])

machine     fan   pump  slider  valve
label                                
abnormal   4425   1368    2670   1437
normal    12225  11247    9612  11073


label          abnormal  normal
machine id                     
fan     id_00      1221    3033
        id_02      1077    3048
        id_04      1044    3099
        id_06      1083    3045
pump    id_00       429    3018
        id_02       333    3015
        id_04       300    2106
        id_06       306    3108
slider  id_00      1068    3204
        id_02       801    3204
        id_04       534    1602
        id_06       267    1602
valve   id_00       357    2973
        id_02       360    2124
        id_04       360    3000
        id_06       360    2976

## 2. Grouping strategy

### Identifying potentially related recordings

The metadata analysis identified 18,019 groups of three recordings, corresponding to the three SNR conditions.

Recordings sharing the same machine type, machine ID, label, and filename are treated as potentially related versions of the same underlying recording.

To reduce the risk of data leakage, all recordings belonging to the same group must be assigned to the same dataset partition.

In [4]:
df['snr'].value_counts()

snr
-6    18019
 0    18019
 6    18019
Name: count, dtype: int64

### Group identifier

A unique group identifier is created by combining:

- `machine`
- `id`
- `label`
- `file`

The `snr` column is deliberately excluded from the identifier so that the three SNR versions remain in the same group.

In [5]:
group_cols = ['machine', 'id', 'label', 'file']

df['grouped_snr_audio'] = df[group_cols].agg(func='_'.join, axis=1)

df.head()

,noise,machine,id,label,file,snr,file_num,grouped_snr_audio
0,-6_dB_fan,fan,id_00,abnormal,00000000.wav,-6,0,fan_id_00_abnormal_00000000.wav
1,-6_dB_fan,fan,id_00,abnormal,00000001.wav,-6,1,fan_id_00_abnormal_00000001.wav
2,-6_dB_fan,fan,id_00,abnormal,00000002.wav,-6,2,fan_id_00_abnormal_00000002.wav
3,-6_dB_fan,fan,id_00,abnormal,00000003.wav,-6,3,fan_id_00_abnormal_00000003.wav
4,-6_dB_fan,fan,id_00,abnormal,00000004.wav,-6,4,fan_id_00_abnormal_00000004.wav


### Grouping validation

The resulting dataset contains:

- 54,057 observations.
- 18,019 unique groups.
- Exactly three observations per group.

**Assumption:** The relationship between SNR versions is inferred from the metadata structure. It has not been independently confirmed through audio signal comparison.

In [6]:
print(df['grouped_snr_audio'].nunique())
df.groupby(by='grouped_snr_audio').size().value_counts()

18019


3    18019
Name: count, dtype: int64


## 3. Stratified group split

### Method

We use `StratifiedGroupKFold` from scikit-learn to generate five folds.

This method combines two constraints:

- **Stratification:** preserving the overall proportion of `normal` and `abnormal` labels as closely as possible.
- **Grouping:** ensuring that observations belonging to the same group are not separated between training and evaluation datasets.

### Configuration

- `n_splits=5`
- `shuffle=True`
- `random_state=42`

### Final partitioning

The five folds are allocated as follows:

| Dataset | Number of folds | Target proportion |
|---|---:|---:|
| Training | 3 | 60% |
| Validation | 1 | 20% |
| Test | 1 | 20% |

The training dataset is constructed by concatenating the indices of three folds.

The remaining two folds are assigned to validation and testing.

The split is performed on metadata before feature extraction, allowing different models to use identical data partitions.


In [7]:
from sklearn.model_selection import StratifiedGroupKFold

X = df.drop(columns=['label', 'grouped_snr_audio'])

y = df['label']

groups = df['grouped_snr_audio']

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

table_test_idx = []
for fold, (train_idx, test_idx) in enumerate(sgkf.split(X, y, groups)):
    print(f"--- Fold {fold + 1} ---")
    print(f"Train : {len(train_idx)} observations")
    print(f"Test         : {len(test_idx)} observations")
    print(f"Class distribution in the test set: {y.iloc[test_idx].value_counts(normalize=True)}\n")
    table_test_idx.append(test_idx)
    group_train_check = set(groups.iloc[train_idx])
    group_test_check = set(groups.iloc[test_idx])
    assert group_train_check.isdisjoint(group_test_check), \
        f"Overlapping groups detected in fold {fold + 1}"

    print(f"Fold {fold + 1}: No overlapping groups detected ✓")

table_test_idx

--- Fold 1 ---
Train : 43245 observations
Test         : 10812 observations
Class distribution in the test set: label
normal      0.81576
abnormal    0.18424
Name: proportion, dtype: float64

Fold 1: No overlapping groups detected ✓
--- Fold 2 ---
Train : 43245 observations
Test         : 10812 observations
Class distribution in the test set: label
normal      0.81687
abnormal    0.18313
Name: proportion, dtype: float64

Fold 2: No overlapping groups detected ✓
--- Fold 3 ---
Train : 43245 observations
Test         : 10812 observations
Class distribution in the test set: label
normal      0.817425
abnormal    0.182575
Name: proportion, dtype: float64

Fold 3: No overlapping groups detected ✓
--- Fold 4 ---
Train : 43245 observations
Test         : 10812 observations
Class distribution in the test set: label
normal      0.816593
abnormal    0.183407
Name: proportion, dtype: float64

Fold 4: No overlapping groups detected ✓
--- Fold 5 ---
Train : 43248 observations
Test         : 10809 o

[array([    0,     6,    10, ..., 54044, 54048, 54053], shape=(10812,)),
 array([    1,     7,    12, ..., 54045, 54049, 54054], shape=(10812,)),
 array([    2,     8,    14, ..., 54043, 54051, 54056], shape=(10812,)),
 array([    4,     5,    11, ..., 54046, 54052, 54055], shape=(10812,)),
 array([    3,     9,    13, ..., 54042, 54047, 54050], shape=(10809,))]

## 4. Split validation

The resulting partitions are evaluated using several checks.

### 4.1 Dataset sizes

| Dataset | Observations | Approximate proportion |
|---|---:|---:|
| Training | 32,436 | 60% |
| Validation | 10,809 | 20% |
| Test | 10,812 | 20% |
| **Total** | **54,057** | **100%** |

In [9]:
train_idx = np.concatenate(table_test_idx[:3])
test_idx = table_test_idx[3]
validate_idx = table_test_idx[4]
print(len(train_idx))
print(len(test_idx))
print(len(validate_idx))

32436
10812
10809


### 4.2 Class distribution

| Dataset | Normal | Abnormal |
|---|---:|---:|
| Training | 81.67% | 18.33% |
| Validation | 81.77% | 18.23% |
| Test | 81.66% | 18.34% |

The overall class distribution is preserved closely across the three datasets.

In [10]:
print(y.iloc[train_idx].value_counts(normalize=True))
print(y.iloc[test_idx].value_counts(normalize=True))
print(y.iloc[validate_idx].value_counts(normalize=True))

label
normal      0.816685
abnormal    0.183315
Name: proportion, dtype: float64
label
normal      0.816593
abnormal    0.183407
Name: proportion, dtype: float64
label
normal      0.817652
abnormal    0.182348
Name: proportion, dtype: float64


### 4.3 Group overlap

Pairwise intersections of group identifiers are checked between:

- Training and validation.
- Training and test.
- Validation and test.

All three intersections are empty.

This confirms that no group identifier is shared between the final partitions.

In [11]:
group_train_check = set(groups.iloc[train_idx])
group_test_check = set(groups.iloc[test_idx])
group_validate_check = set(groups.iloc[validate_idx])
print(group_train_check.intersection(group_test_check))
print(group_train_check.intersection(group_validate_check))
print(group_test_check.intersection(group_validate_check))

set()
set()
set()


### 4.4 Anomaly distribution by machine type

| Machine | Full dataset | Training | Validation | Test |
|---|---:|---:|---:|---:|
| Fan | 26.58% | 26.55% |  26.58% | 26.67% |
| Pump | 10.84% | 10.82% | 10.70% | 11.06% |
| Slider | 21.74% | 21.73% | 21.86% | 21.64% |
| Valve | 11.49% | 11.63% | 11.16% | 11.38% |

Values represent the percentage of abnormal observations within each machine type.

The anomaly proportions remain similar across the three partitions.

In [12]:
print(pd.crosstab(df['machine'], df['label'], normalize='index'))
print(pd.crosstab(df['machine'].iloc[train_idx], df['label'].iloc[train_idx], normalize='index'))
print(pd.crosstab(df['machine'].iloc[test_idx], df['label'].iloc[test_idx], normalize='index'))
print(pd.crosstab(df['machine'].iloc[validate_idx], df['label'].iloc[validate_idx], normalize='index'))

label    abnormal    normal
machine                    
fan      0.265766  0.734234
pump     0.108442  0.891558
slider   0.217391  0.782609
valve    0.114868  0.885132
label    abnormal    normal
machine                    
fan      0.265465  0.734535
pump     0.108205  0.891795
slider   0.217338  0.782662
valve    0.116307  0.883693
label    abnormal    normal
machine                    
fan      0.266667  0.733333
pump     0.110583  0.889417
slider   0.216381  0.783619
valve    0.113772  0.886228
label    abnormal    normal
machine                    
fan      0.265766  0.734234
pump     0.107015  0.892985
slider   0.218559  0.781441
valve    0.111645  0.888355


### Conclusion

The split satisfies the tested requirements for group separation, overall class balance, and machine-level class distribution.


## 5. Dataset export

Three metadata DataFrames are created using the selected row indices:

- `df_train`
- `df_validate`
- `df_test`

Each DataFrame retains the original metadata columns and the group identifier.

The resulting datasets are exported as separate CSV files.

These files define the data partitions to be used during downstream feature extraction and model development.

The audio files themselves are not modified, duplicated, or downloaded during this process.

The CSV files can subsequently be made available through the project's shared data storage.


In [14]:
df_train = df.iloc[train_idx].copy()
df_validate = df.iloc[validate_idx].copy()
df_test = df.iloc[test_idx].copy()

In [15]:
df_train.to_csv('../raw_data/df_train.csv', index=False)
df_validate.to_csv('../raw_data/df_validate.csv', index=False)
df_test.to_csv('../raw_data/df_test.csv', index=False)


## 6. Limitations and future considerations

### Grouping assumption

The grouping strategy assumes that recordings sharing the same machine type, machine ID, label, and filename across different SNR conditions are related.

This assumption is supported by the metadata structure but has not been verified through direct audio comparison.

### Generalization to unseen machine IDs

The current strategy prevents overlap between defined recording groups, but it does not require machine IDs to be exclusive to one partition.

Therefore, this split is not designed to evaluate generalization to entirely unseen machine IDs.

A separate machine-ID-based evaluation strategy would be required for that objective.

### Other potential sources of data leakage

The absence of overlapping group identifiers does not guarantee the absence of every possible source of data leakage.

Any preprocessing step that estimates parameters from data, such as feature scaling, must be fitted using the training dataset only.

The validation and test datasets must remain separate from model fitting and hyperparameter optimization.

### Reproducibility

The splitting configuration uses a fixed random seed (`random_state=42`).

The splitting code, grouping definition, dataset ordering, and dependency versions should be preserved to reproduce the same partitions.

The exported CSV files provide a fixed reference for subsequent model experiments.
